In [1]:
# ============================================================
# SANKEY CHART — Tamil Nadu Seat Flips 2021 to 2026
# Purpose: Visualise which party held each seat in 2021
#          and which party won it in 2026
# Author: Aditya Dixit
# Source: Codebasics RPC — ECI Data
# ============================================================

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# ------------------------------------------------------------
# STEP 1 — LOAD DATA
# Each row represents one constituency
# Y_2021 = winning party in 2021
# Y_2026 = winning party in 2026
# ------------------------------------------------------------
df = pd.read_excel(r"D:\TAMIL NADU PROJECT\SANKEY CHART DATA.xlsx")

# Quick sanity check — verify column names and first few rows
print(df.columns.tolist())
print(df.head())

# ------------------------------------------------------------
# STEP 2 — COUNT FLOWS
# Group by 2021 party → 2026 party combination
# Count how many constituencies followed each flow path
# Example: DMK-2021 → TVK-2026 = 65 constituencies
# ------------------------------------------------------------
flows = df.groupby(["Y_2021", "Y_2026"]).size().reset_index(name="count")

# ------------------------------------------------------------
# STEP 3 — BUILD NODE LIST
# Nodes are all unique parties appearing in either year
# pd.unique flattens both columns into one list, removes duplicates
# node_index maps each party name to a numeric index
# Plotly Sankey uses numeric indices, not party names directly
# ------------------------------------------------------------
all_nodes = list(pd.unique(flows[["Y_2021", "Y_2026"]].values.ravel()))
node_index = {node: i for i, node in enumerate(all_nodes)}

# ------------------------------------------------------------
# STEP 4 — BUILD SOURCE, TARGET, VALUE LISTS
# Plotly Sankey needs three parallel lists:
# source[i] = index of the left node for flow i
# target[i] = index of the right node for flow i
# value[i]  = thickness of the ribbon = number of constituencies
# ------------------------------------------------------------
source = [node_index[s] for s in flows["Y_2021"]]
target = [node_index[t] for t in flows["Y_2026"]]
value = flows["count"].tolist()

# ------------------------------------------------------------
# STEP 5 — BUILD NODE LABELS WITH SEAT COUNTS
# Default label is just the party name
# We append the total seats won/lost below the party name
# <br> is HTML line break — renders inside Plotly labels
# Logic:
#   If node appears in Y_2021 → sum all seats it sent out
#   If node appears only in Y_2026 → sum all seats it received
# ------------------------------------------------------------
node_labels = []
for node in all_nodes:
    if node in flows["Y_2021"].values:
        # Party existed in 2021 — show how many seats it held
        total = flows[flows["Y_2021"] == node]["count"].sum()
        node_labels.append(f"{node}<br>{total} seats")
    elif node in flows["Y_2026"].values:
        # Party only appears in 2026 — show how many seats it won
        total = flows[flows["Y_2026"] == node]["count"].sum()
        node_labels.append(f"{node}<br>{total} seats")
    else:
        # Fallback — just show party name
        node_labels.append(node)

# ------------------------------------------------------------
# STEP 6 — ASSIGN NODE COLOURS
# Each party gets a consistent colour across both sides
# Colours are chosen for visual distinction only
# They do NOT represent official party colours or ideology
# ------------------------------------------------------------
node_colors = []
for node in all_nodes:
    if "TVK" in node:
        node_colors.append("#378ADD")    # blue
    elif "DMK" in node:
        node_colors.append("#1D9E75")    # green
    elif "AIADMK" in node:
        node_colors.append("#D85A30")    # orange
    elif "INC" in node:
        node_colors.append("#A0522D")    # brown
    elif "BJP" in node:
        node_colors.append("#FF9933")    # saffron
    elif "PMK" in node:
        node_colors.append("#8B008B")    # purple
    elif "VCK" in node:
        node_colors.append("#4682B4")    # steel blue
    else:
        node_colors.append("#888780")    # grey for all others

# ------------------------------------------------------------
# STEP 7 — ASSIGN LINK (RIBBON) COLOURS
# Links are coloured by the SOURCE party (2021 side)
# rgba format: rgba(R, G, B, opacity)
# Low opacity (0.2-0.3) keeps ribbons readable without clutter
# ------------------------------------------------------------
link_colors = []
for s in flows["Y_2021"]:
    if "DMK" in s:
        link_colors.append("rgba(29,158,117,0.3)")   # green tint
    elif "AIADMK" in s:
        link_colors.append("rgba(216,90,48,0.3)")    # orange tint
    elif "INC" in s:
        link_colors.append("rgba(160,82,45,0.3)")    # brown tint
    elif "BJP" in s:
        link_colors.append("rgba(255,153,51,0.3)")   # saffron tint
    else:
        link_colors.append("rgba(180,178,169,0.2)")  # grey tint

# ------------------------------------------------------------
# STEP 8 — BUILD THE SANKEY FIGURE
# arrangement="snap" keeps nodes aligned to left and right
# pad = spacing between nodes vertically
# thickness = width of each node bar
# link color uses our custom link_colors list
# ------------------------------------------------------------
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=20,                          # vertical spacing between nodes
        thickness=25,                    # width of node bars
        line=dict(color="white", width=0.5),  # thin white border on nodes
        label=node_labels,               # party name + seat count
        color=node_colors                # party colours
    ),
    link=dict(
        source=source,                   # left side node indices
        target=target,                   # right side node indices
        value=value,                     # ribbon thickness = seat count
        color=link_colors                # ribbon colours by source party
    )
)])

# ------------------------------------------------------------
# STEP 9 — LAYOUT AND STYLING
# Clean white background to match minimalist deck aesthetic
# Font size 13 for readability in presentation context
# ------------------------------------------------------------
fig.update_layout(
    title_text="Seat Flips — 2021 to 2026",
    font=dict(size=13, family="Arial"),
    paper_bgcolor="white",              # outer background
    plot_bgcolor="white",               # chart area background
    width=1200,                         # pixel width for export
    height=650                          # pixel height for export
)

# ------------------------------------------------------------
# STEP 10 — EXPORT AND DISPLAY
# Saves as interactive HTML file — open in Chrome for full quality
# Take screenshot from browser for PPT slide
# fig.show() opens in default browser automatically
# ------------------------------------------------------------
pio.write_html(fig, r"D:\TAMIL NADU PROJECT\sankey_chart.html")
fig.show()

['Y_2021', 'Y_2026', 'CONSTITUENCY', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18']
     Y_2021       Y_2026  CONSTITUENCY  Unnamed: 3  Unnamed: 4  \
0  DMK-2021     TVK-2026             1         NaN         NaN   
1  INC-2021     TVK-2026             2         NaN         NaN   
2  DMK-2021  AIADMK-2026             3         NaN         NaN   
3  DMK-2021     TVK-2026             4         NaN         NaN   
4  DMK-2021     TVK-2026             5         NaN         NaN   

              Unnamed: 5     Unnamed: 6 Unnamed: 7 Unnamed: 8   Unnamed: 9  \
0  Count of CONSTITUENCY  Column Labels        NaN        NaN          NaN   
1             Row Labels    AIADMK-2026  AMMK-2026   BJP-2026  CPI(M)-2026   
2            AIADMK-2021             22        NaN        NaN          NaN   
3               BJP-2

c:\Users\Aditya Dixit\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.


